# D3 — Counterfactual explanations

**L3 (SIMULATED, the primary part of D3, because only L3 has actionable context):** DiCE ("random", 4 counterfactuals) for 100 held-out findings that the L3 model flags High/Critical. Only remediation levers may vary: `internet_facing`, `has_admin_path`, `admin_principal_count`, `principal_count`. Each counterfactual is checked for validity under the model **and under the true label formula**, using a re-scorer that reproduces `priority_score` to 1.7e-16.

**L1:** a CVE's CVSS score and description never change, so the only meaningful what-if is EPSS. For each E-rand CVE outside its month's top-100, the minimum point-in-time EPSS that lifts its M-XGB score to τ₁₀₀ is found by a deterministic search over 200 log-spaced values. The EPSS percentile follows from that scoring date's empirical EPSS→percentile map. Design: Study D addendum (commit c0a3ee1), registered before any Study D explanation was computed.

Reads saved outputs. Run: `python -m src.explainability.counterfactuals` (~2 min).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

## L3 — SIMULATED: DiCE with remediation levers

In [2]:
pd.read_csv("outputs/d3_l3_summary.csv").T

,0
part,L3 (SIMULATED) DiCE
n_findings,100
share_with_any_cf,0.36
share_with_model_valid_cf,0.36
n_cf,144
share_cf_valid_model,1.0
share_model_valid_also_formula_valid,0.965278
share_model_valid_iam_only,0.006944
share_iam_only_formula_valid,0.0
share_model_valid_using_internet_facing,0.993056


In [3]:
cf = pd.read_csv("outputs/d3_l3_counterfactuals.csv")
cf.changed.value_counts()

changed
internet_facing,principal_count          94
internet_facing,admin_principal_count    49
has_admin_path,admin_principal_count      1
Name: count, dtype: int64

**Reading (L3, simulated).**
- **36 of 100 flagged findings can be de-escalated by a lever an engineer controls**, and every such counterfactual removes internet exposure. Only 46 of the 100 were internet-facing at all, so about 78% of those can be fixed this way. For the rest no lever in the set changes the label, which is correct: the formula uses no IAM feature.
- **Validity under the true formula: 96.5%.** Model and formula agree on almost every counterfactual. The one IAM-only counterfactual the model accepted is **invalid under the formula**. This is exactly the model artefact predicted in the pre-registration: the model learned a tiny spurious dependence on IAM features.
- **Sparsity failure (exploratory check, added after a 5-finding pilot):** 99.3% of valid counterfactuals change `internet_facing` *plus* an irrelevant IAM count, even though flipping `internet_facing` alone would have been enough. DiCE's random method does not minimise the number of changes, so its suggestions over-prescribe work. A real recommendation needs a minimality post-processing step.

## L1 — what EPSS would put this CVE in the monthly top-100?

In [4]:
pd.read_csv("outputs/d3_l1_summary.csv").T

,0
part,L1 EPSS trigger
tau100,0.006519
n_cves,974
share_reachable,0.989733
epss_now_median,0.00051
epss_needed_median,0.020655
epss_needed_q25,0.013481
epss_needed_q75,0.034206
pct_needed_median,0.844665
uplift_ratio_median,28.133042


In [5]:
t = pd.read_csv("outputs/d3_l1_epss_triggers.csv")
t[["epss_now", "epss_needed", "pct_needed", "uplift_ratio"]].describe().round(4)

,epss_now,epss_needed,pct_needed,uplift_ratio
count,974.0000,964.0000,964.0000,964.0000
mean,0.0017,0.0226,0.8095,48.7294
std,0.0024,0.0117,0.1102,70.8006
min,0.0000,0.0004,0.0292,1.0000
25%,0.0004,0.0135,0.7727,9.2983
50%,0.0005,0.0207,0.8447,28.1330
75%,0.0023,0.0342,0.8743,64.7442
max,0.0241,0.0929,0.9390,915.0123


**Reading (L1).** τ₁₀₀ = 0.0065. For 99% of the CVEs outside the top-100, *some* EPSS value would lift them in. The median trigger is **EPSS ≈ 0.021**, about the 84th percentile on its scoring date, against a current median of 0.0005, a 28× rise. This gives a concrete, monitorable statement per CVE: "escalate if EPSS crosses 0.02". It is a natural use of counterfactuals when the finding itself cannot be changed, only watched.